# shannon_model — entrenamiento en Google Colab

1. Runtime → Change runtime type → **GPU**
2. Ejecuta las celdas en orden
3. Ajusta `REPO_URL` / `BRANCH` si hace falta

## 1. Configurar repo

In [ ]:
# Cambia estos valores si el remoto o la rama son distintos
REPO_URL = "https://github.com/isaacgrimaldovo/shannon_model.git"
BRANCH = "main"
REPO_DIR = "shannon_model"

# Si el repo es privado, descomenta y usa un secret de Colab:
# from google.colab import userdata
# TOKEN = userdata.get("GITHUB_TOKEN")
# REPO_URL = f"https://{TOKEN}@github.com/isaacgrimaldovo/shannon_model.git"

In [ ]:
import os
from pathlib import Path

if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print("cwd:", os.getcwd())

## 2. (Opcional) Montar Google Drive para checkpoints/datos

In [ ]:
USE_DRIVE = False  # True para guardar en Drive

if USE_DRIVE:
    from google.colab import drive
    from pathlib import Path

    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive/shannon_model")
    (drive_root / "checkpoints").mkdir(parents=True, exist_ok=True)
    (drive_root / "data" / "raw").mkdir(parents=True, exist_ok=True)
    print("Drive listo en", drive_root)

## 3. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt

import sys
from pathlib import Path

sys.path.insert(0, str(Path("src").resolve()))
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 4. Entrenar

In [ ]:
from shannon_model.train import run_training

CONFIG = "configs/default.yaml"
result = run_training(CONFIG)
result

## 5. Poll loop (auto-run en cambios push)

Flujo: hacé `commit` + `push` desde tu entorno local tocando `src/`, `scripts/` o `configs/`.
Con este loop corriendo en Colab, cada `POLL_INTERVAL` segundos se hace `git pull`;
si detecta un commit nuevo que toca esos paths, dispara `scripts/train.py` automáticamente
(mismo entrypoint que la celda de arriba). Cambios que solo tocan docs/README/notebook no disparan run.

Resultados quedan en `checkpoints/` (o en Drive si `USE_DRIVE=True`, ver seccion 2).

Para parar: botón **Interrupt execution** de Colab — el loop corta limpio, sin dejar el training colgado.

In [ ]:
import subprocess
import time

POLL_INTERVAL = 20  # segundos entre cada git pull
WATCH_PATHS = ("src/", "scripts/", "configs/")


def _current_sha():
    return subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    ).stdout.strip()


def _relevant_change(old_sha, new_sha):
    diff = subprocess.run(
        ["git", "diff", "--name-only", old_sha, new_sha],
        capture_output=True, text=True, check=True,
    ).stdout.splitlines()
    return any(path.startswith(WATCH_PATHS) for path in diff)


last_sha = _current_sha()
print(f"[poll] loop iniciado, SHA inicial: {last_sha[:8]}")

try:
    while True:
        subprocess.run(["git", "pull"], capture_output=True, text=True)
        new_sha = _current_sha()

        if new_sha == last_sha:
            print(f"[poll] {new_sha[:8]} — sin cambios")
        elif _relevant_change(last_sha, new_sha):
            print(f"[poll] {new_sha[:8]} — cambio relevante, corriendo scripts/train.py")
            subprocess.run(["python", "scripts/train.py", "--config", CONFIG], check=False)
            last_sha = new_sha
        else:
            print(f"[poll] {new_sha[:8]} — cambio detectado pero irrelevante (no toca src/scripts/configs)")
            last_sha = new_sha

        time.sleep(POLL_INTERVAL)
except KeyboardInterrupt:
    print("[poll] interrumpido manualmente, loop detenido")

## 6. (Opcional) Copiar mejor checkpoint a Drive

In [ ]:
import shutil
from pathlib import Path

if USE_DRIVE:
    src = Path("checkpoints/best.pt")
    dst = Path("/content/drive/MyDrive/shannon_model/checkpoints/best.pt")
    if src.exists():
        shutil.copy2(src, dst)
        print("Copiado a", dst)
    else:
        print("No existe", src)
else:
    print("USE_DRIVE=False — checkpoints en", Path("checkpoints").resolve())